In [16]:
model = "llama3.2:1b"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [17]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model)

In [18]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
)

In [19]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection("ai_model_book")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name="ai_model_book",
    embedding_function=embedding_model,
)

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#ADD
chain = {"docs": format_docs} | prompt | model | StrOutputParser()

In [21]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query)

print(docs)

[Document(metadata={'page': 33, 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\n Whether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\n Whether or not they can learn incrementally on the fly (online versus batch\nlearning)\n Whether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and build a predictive model, much\nlike scientists do (instance-based versus model-based learning)\nThese criteria are not exclusive; you can combine them in any way you like. For\nexample, a state-of-the-art spam filter may learn on the fly using a deep neural net\nwork model trained using examples of spam and ham; this makes it an online, model-\nbased, supervised learning system.\nLets look at each of these cr

In [22]:
chain.invoke(docs)

'The main themes in these retrieved docs related to the types of Machine Learning systems are:\n\n1. **Classification**: The docs discuss how machine learning systems can be categorized based on their classification methods, including supervised, unsupervised, semisupervised, and Reinforcement Learning.\n2. **Learning Paradigms**: The docs also explore different learning paradigms in machine learning, such as instance-based vs model-based learning, online vs batch learning, and incremental learning.\n3. **Generalization**: The importance of generalization is highlighted, where the system needs to perform well on new instances or data sets that it has not seen before.\n4. **Classification Algorithms**: Various classification algorithms are discussed, including supervised vs unsupervised, instance-based vs model-based learning, and different measures of similarity used in instance-based learning (e.g. word count).\n5. **Reinforcement Learning**: The docs mention Reinforcement Learning as

In [24]:
# Simple stream the chain output
for chunk in chain.stream(docs):
    print(chunk, end="", flush=True)

Based on the retrieved documents, the main themes in these types of machine learning systems are:

1. **Classification**: Many machine learning systems are used for classification tasks, where they need to identify patterns or classify data into specific categories (e.g., spam vs. non-spam emails).
2. **Supervised and Unsupervised Learning**: Machine learning systems can be classified based on the amount and type of supervision they receive during training, including supervised, unsupervised, semisupervised, and reinforcement learning.
3. **Generalization**: Most machine learning tasks require generalization to new instances or data that has not been seen before. This is achieved through various techniques such as instance-based and model-based learning.
4. **Learning Methods**: Machine learning systems can be categorized based on their learning methods, including batch and online learning, where they may learn incrementally from a stream of incoming data.

Some key concepts in machine

In [26]:
# More complex async event streaming
async for event in chain.astream_events(docs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Based on the retrieved documents, the main themes in the category "Types of Machine Learning Systems" are:

1. **Classification**: The system is classified into different types based on its characteristics, such as whether it's trained with human supervision, can learn incrementally online or offline, and how it generalizes to new instances.
2. **Learning methods**: The document discusses various machine learning techniques, including:
	* Supervised learning: the algorithm learns from labeled data
	* Instance-based versus model-based learning: different approaches to generalization
	* Batch and online learning: whether the system can learn incrementally from a stream of incoming data
3. **Generalization**: The document emphasizes that most machine learning tasks require the system to generalize well to new, unseen examples.
4. **Categorization criteria**: The document outlines four major categories for classifying machine learning systems:
	* Supervised/Unsupervised Learning: classific

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [27]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever()

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [28]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x126a72d10>)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"))])
| ChatOllama(model='llama3.2:1b', _client=<ollama._client.Client object at 0x133cc6ad0>, _async_client=<ollama._client.AsyncClient object at 0x1277a1410>)
| StrOutputParser()

In [29]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning where the training data includes the desired solutions or labels, and the algorithm learns to map inputs (X) to outputs (y). The system tries to learn without a teacher by trying to predict the output for each input, given the label.'

In [30]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where the training data includes the desired solutions (labels), and the algorithm tries to learn without a teacher, by trying to fit the patterns in the training data.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [31]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [32]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [33]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [34]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, which means that the data is already classified or tagged with the correct output or response. The goal of supervised learning is to learn a mapping between input features (also known as inputs or observations) and the corresponding labels or outputs.

In other words, in supervised learning, you have:

1. **Labeled data**: You have a dataset that includes examples where some input feature is labeled with a particular output or response.
2. **Predictive model**: The algorithm learns to predict the output or response for new, unseen input features by analyzing the patterns and relationships in the labeled data.

For example, consider a classification problem:

* You want to train an algorithm to classify images of cats and dogs as either "cat" or "dog".
* The labeled dataset would include examples such as:
	+ Cat: [image 1, label 0]
	+ Dog: [image 2, label 1]
	+ Bird: [image 3, label 2]

In s

In [35]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased follow-up question:

What types of algorithms are commonly used in supervised learning?In supervised learning, some common algorithms include:

1. k-Nearest Neighbors (KNN): This algorithm works by finding the k most similar data points to a new, unseen example and using their similarity as a measure of how well it fits the training data.
2. Linear Regression: This algorithm is used for regression problems where the goal is to predict a continuous output variable based on one or more input variables.
3. Logistic Regression: This algorithm is used for classification problems where the goal is to predict an output variable based on one or more input variables, and it uses binary classifiers (0 or 1) to make predictions.
4. Support Vector Machines (SVMs): This algorithm is used for both regression and classification problems, and it works by finding the hyperplane that maximally separates the training data into different classes.
5. Decision Trees: This algorithm is 